# Gold Layer — modelo analítico de chamados

Neste notebook vamos ler as tabelas da camada Silver e criar as tabelas da camada Gold. Nesta etapa organizamos os dados em dimensões, consolidamos custos e pesquisas no grão de cada chamado, construímos a fato principal e geramos a visão executiva mensal que será usada nas análises e no painel.

Como os arquivos recebidos representam cargas completas e mais de uma pessoa do grupo pode executar o notebook, usamos `overwrite` para que as reexecuções não acumulem registros duplicados.

## Regras e premissas

- **Grão da fato:** uma linha por `id_chamado`. Custos e pesquisas são consolidados antes dos joins.
- **Referência temporal:** `hora_abertura_chamado`, proveniente de `Chamados_Hora.CSV` e tratada na Silver. Chamados sem data válida permanecem na fato.
- **Resolução:** o denominador contém apenas `Sim` e `Não`; `Não informado` é preservado e excluído do indicador. Decisão do projeto registrada em 28/10/2026.
- **Satisfação:** notas de 4 ou 5 são respostas satisfeitas; somente notas entre 1 e 5 são elegíveis. A referência histórica de 85% só é comparável se a empresa confirmar a mesma fórmula.
- **Tempos:** apenas tempos recalculados e marcados como válidos entram nas médias.
- **Custo ausente:** permanece nulo, pois ausência de informação não equivale a custo zero.
- **Funil:** URA e Chatbot são Autosserviço; Atendimento inicial é 1º nível; Atendimento especializado é 2º nível. Prevenção não é criada porque não aparece nas bases.
- **Dados cadastrais:** os dados de clientes e atendentes são mantidos na Gold. Os atributos dos clientes ficam concentrados em `dim_clientes` e são relacionados à fato por `id_cliente`.

## Configuração

O notebook de configuração centraliza catálogo e schemas. Os nomes das tabelas ficam declarados nesta célula para evitar strings dispersas.

In [0]:
%run ./00_Config

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

tabelas_silver = {
    "chamados": f"{silver_schema}.tb_chamados",
    "canais": f"{silver_schema}.tb_canais",
    "motivos": f"{silver_schema}.tb_motivos",
    "atendentes": f"{silver_schema}.tb_atendentes",
    "clientes": f"{silver_schema}.tb_clientes",
    "custos": f"{silver_schema}.tb_custos",
    "pesquisas": f"{silver_schema}.tb_pesquisa_satisfacao",
}

tabelas_gold = {
    "canais": f"{gold_schema}.dim_canais",
    "motivos": f"{gold_schema}.dim_motivos",
    "atendentes": f"{gold_schema}.dim_atendentes",
    "clientes": f"{gold_schema}.dim_clientes",
    "data": f"{gold_schema}.dim_data",
    "chamados": f"{gold_schema}.fato_chamados",
    "mensal": f"{gold_schema}.agg_visao_executiva_mensal",
}

## Leitura e validações de entrada

As leituras selecionam somente os atributos necessários. Unicidade do chamado e das referências, chaves obrigatórias e domínios críticos interrompem a execução antes de qualquer persistência.

In [0]:
df_silver_chamados = spark.table(tabelas_silver["chamados"]).select(
    "id_chamado", "id_cliente", "id_canal", "id_motivo", "id_atendente",
    "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento",
    "resolvido", "tempo_espera_calculado_segundos",
    "tempo_atendimento_calculado_segundos", "fl_espera_valida",
    "fl_atendimento_valido", "fl_ciclo_atendimento_valido",
    "fl_atendente_cadastrado",
)
df_silver_canais = spark.table(tabelas_silver["canais"])
df_silver_motivos = spark.table(tabelas_silver["motivos"])
df_silver_atendentes = spark.table(tabelas_silver["atendentes"])
df_silver_clientes = spark.table(tabelas_silver["clientes"]).select(
    "id_cliente", "nome", "email", "regiao", "idade",
    "data_cadastro", "dia_vencimento_fatura",
)
df_silver_custos = spark.table(tabelas_silver["custos"]).select("id_custo", "id_chamado", "custo")
df_silver_pesquisas = spark.table(tabelas_silver["pesquisas"]).select("id_pesquisa", "id_chamado", "nota_atendimento", "nota_valida")

def validar_chave_unica(df: DataFrame, nome: str, chave: str) -> None:
    """Interrompe a execução quando uma chave obrigatória é nula ou repetida."""
    qtd_linhas = df.count()
    qtd_chaves = df.select(chave).distinct().count()
    qtd_nulas = df.filter(F.col(chave).isNull()).count()
    assert qtd_nulas == 0, f"{nome} inválida: {qtd_nulas} registros sem {chave}."
    assert qtd_linhas == qtd_chaves, f"{nome} inválida: {chave} não é único."

validar_chave_unica(df_silver_chamados, tabelas_silver["chamados"], "id_chamado")
validar_chave_unica(df_silver_canais, tabelas_silver["canais"], "id_canal")
validar_chave_unica(df_silver_motivos, tabelas_silver["motivos"], "id_motivo")
validar_chave_unica(df_silver_atendentes, tabelas_silver["atendentes"], "id_atendente")
validar_chave_unica(df_silver_clientes, tabelas_silver["clientes"], "id_cliente")

qtd_resolucoes_invalidas = df_silver_chamados.filter(~F.col("resolvido").isin("Sim", "Não", "Não informado")).count()
assert qtd_resolucoes_invalidas == 0, f"Gold interrompida: {qtd_resolucoes_invalidas} resoluções fora do domínio esperado."

## Construção das dimensões

Cada dimensão preserva a chave estável da Silver. `dim_data` cobre todas as datas entre a menor e a maior abertura válida; datas inválidas não eliminam chamados da fato.

In [0]:
df_dim_canais = (
    df_silver_canais
    .select("id_canal", "nome_canal", "tipo_canal", "status_canal")
    .withColumn(
        "etapa_funil",
        F.when(F.lower(F.col("nome_canal")).isin("ura", "chatbot"), "Autosserviço")
        .when(F.lower(F.col("nome_canal")) == "atendimento inicial", "1º nível")
        .when(F.lower(F.col("nome_canal")) == "atendimento especializado", "2º nível")
        .otherwise(F.lit(None).cast("string")),
    )
)
df_dim_motivos = (
    df_silver_motivos
    .withColumn("criticidade", F.initcap(F.col("criticidade")))
    .select("id_motivo", "nome_motivo", "categoria", "criticidade"))
df_dim_atendentes = df_silver_atendentes.select("id_atendente", "nome_atendente", "nivel_atendimento", "data_admissao", "data_desligamento", "status_atendente")
df_dim_clientes = (
    df_silver_clientes
    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade").between(0, 17), "Até 17")
        .when(F.col("idade").between(18, 29), "18 a 29")
        .when(F.col("idade").between(30, 44), "30 a 44")
        .when(F.col("idade").between(45, 59), "45 a 59")
        .when(F.col("idade") >= 60, "60 ou mais")
        .otherwise("Não informada"),
    )
)

df_limites_data = df_silver_chamados.agg(
    F.min(F.to_date("hora_abertura_chamado")).alias("data_minima"),
    F.max(F.to_date("hora_abertura_chamado")).alias("data_maxima"),
)
assert df_limites_data.filter(F.col("data_minima").isNull() | F.col("data_maxima").isNull()).count() == 0, "Gold interrompida: não há intervalo de datas de abertura válido."

df_dim_data = (
    df_limites_data
    .select(F.explode(F.sequence("data_minima", "data_maxima", F.expr("interval 1 day"))).alias("data"))
    .withColumn("ano", F.year("data"))
    .withColumn("semestre", F.when(F.month("data") <= 6, 1).otherwise(2))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("mes", F.date_format("data", "MMMM"))
    .withColumn("numero_mes", F.month("data"))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana", F.dayofweek("data"))
    .withColumn("nome_dia_semana", F.date_format("data", "EEEE"))
    .withColumn("fl_fim_de_semana", F.dayofweek("data").isin(1, 7))
)

validar_chave_unica(df_dim_canais, tabelas_gold["canais"], "id_canal")
validar_chave_unica(df_dim_motivos, tabelas_gold["motivos"], "id_motivo")
validar_chave_unica(df_dim_atendentes, tabelas_gold["atendentes"], "id_atendente")
validar_chave_unica(df_dim_clientes, tabelas_gold["clientes"], "id_cliente")
validar_chave_unica(df_dim_data, tabelas_gold["data"], "data")
assert df_dim_canais.filter(F.col("etapa_funil").isNull()).count() == 0, "Gold inválida: existe canal sem etapa do funil explicitamente mapeada."

## Consolidação de custos e pesquisas

**Grão das duas saídas intermediárias:** uma linha por `id_chamado`.

Custos preservam soma decimal e quantidade de registros. Pesquisas preservam numeradores e denominadores aditivos, evitando média de médias em qualquer agrupamento posterior.

In [0]:
df_custos_por_chamado = (
    df_silver_custos
    .groupBy("id_chamado")
    .agg(
        F.sum("custo").cast("decimal(18,2)").alias("custo_total"),
        F.countDistinct("id_custo").alias("qtd_registros_custo"),
    )
)

df_pesquisas_por_chamado = (
    df_silver_pesquisas
    .groupBy("id_chamado")
    .agg(
        F.countDistinct("id_pesquisa").alias("qtd_pesquisas_recebidas"),
        F.sum(F.when(F.col("nota_valida"), 1).otherwise(0)).alias("qtd_pesquisas_validas"),
        F.sum(F.when(F.col("nota_valida"), F.col("nota_atendimento")).otherwise(0)).alias("soma_notas_validas"),
        F.sum(F.when(F.col("nota_valida") & F.col("nota_atendimento").isin(4, 5), 1).otherwise(0)).alias("qtd_respostas_satisfeitas"),
    )
)

## Construção da fato de chamados

Os lados potencialmente 1:N já estão consolidados. Os joins são `left` para preservar todos os chamados. A fato mantém `id_cliente` para ligação com a dimensão completa e replica somente região e faixa etária, que são usadas nos agrupamentos analíticos.

In [0]:
df_clientes_analitico = df_dim_clientes.select("id_cliente", "regiao", "faixa_etaria")

df_fato_chamados = (
    df_silver_chamados.alias("ch")
    .join(df_dim_canais.alias("ca"), on="id_canal", how="left")
    .join(df_dim_motivos.alias("mo"), on="id_motivo", how="left")
    .join(df_dim_atendentes.alias("at"), on="id_atendente", how="left")
    .join(df_clientes_analitico.alias("cl"), on="id_cliente", how="left")
    .join(df_custos_por_chamado.alias("cu"), on="id_chamado", how="left")
    .join(df_pesquisas_por_chamado.alias("pe"), on="id_chamado", how="left")
    .withColumn("fl_resolucao_informada", F.col("resolvido").isin("Sim", "Não"))
    .withColumn("fl_resolvido", F.col("resolvido") == "Sim")
    .withColumn("fl_custo_disponivel", F.col("custo_total").isNotNull())
    .withColumn("fl_pesquisa_recebida", F.col("qtd_pesquisas_recebidas").isNotNull())
    .withColumn("fl_pesquisa_valida", F.coalesce(F.col("qtd_pesquisas_validas"), F.lit(0)) > 0)
    .withColumn("data_abertura", F.to_date("hora_abertura_chamado"))
    .withColumn("ano_abertura", F.year("hora_abertura_chamado"))
    .withColumn("mes_abertura", F.month("hora_abertura_chamado"))
    .withColumn("ano_mes_abertura", F.date_format("hora_abertura_chamado", "yyyy-MM"))
    .withColumn("dia_semana_abertura", F.dayofweek("hora_abertura_chamado"))
    .withColumn("hora_do_dia_abertura", F.hour("hora_abertura_chamado"))
    .select(
        "id_chamado", "id_cliente", "id_canal", "id_motivo", "id_atendente",
        "nome_canal", "tipo_canal", "etapa_funil", "nome_motivo",
        "categoria", "criticidade", "nome_atendente", "nivel_atendimento", "status_atendente",
        "regiao", "faixa_etaria", "hora_abertura_chamado",
        "hora_inicio_atendimento", "hora_finalizacao_atendimento", "data_abertura",
        "ano_abertura", "mes_abertura", "ano_mes_abertura",
        "dia_semana_abertura", "hora_do_dia_abertura", "resolvido",
        "fl_resolucao_informada", "fl_resolvido", "tempo_espera_calculado_segundos",
        "tempo_atendimento_calculado_segundos", "fl_espera_valida",
        "fl_atendimento_valido", "fl_ciclo_atendimento_valido",
        "fl_atendente_cadastrado", "custo_total", "qtd_registros_custo",
        "fl_custo_disponivel", "qtd_pesquisas_recebidas", "qtd_pesquisas_validas",
        "soma_notas_validas", "qtd_respostas_satisfeitas",
        "fl_pesquisa_recebida", "fl_pesquisa_valida",
    )
)

## Validações da fato

As verificações reconciliam grão, quantidade de chamados, custos e pesquisas. Canal e motivo sem correspondência são falhas críticas; ausência de atendente, custo, pesquisa ou data é mantida e quantificada como cobertura.

In [0]:
qtd_chamados_silver = df_silver_chamados.count()
qtd_chamados_gold = df_fato_chamados.count()
qtd_ids_gold = df_fato_chamados.select("id_chamado").distinct().count()

assert qtd_chamados_gold == qtd_chamados_silver, f"Gold inválida: {qtd_chamados_gold} linhas para {qtd_chamados_silver} chamados Silver."
assert qtd_ids_gold == qtd_chamados_gold, "Gold inválida: fato_chamados não está no grão de um registro por chamado."
assert df_fato_chamados.filter(F.col("id_canal").isNull() | F.col("nome_canal").isNull()).count() == 0, "Gold inválida: chamado sem canal de referência."
assert df_fato_chamados.filter(F.col("id_motivo").isNull() | F.col("nome_motivo").isNull()).count() == 0, "Gold inválida: chamado sem motivo de referência."
assert isinstance(df_fato_chamados.schema["custo_total"].dataType, DecimalType), "Gold inválida: custo_total não é decimal."
assert df_fato_chamados.filter(F.col("qtd_respostas_satisfeitas") > F.col("qtd_pesquisas_validas")).count() == 0, "Gold inválida: respostas satisfeitas excedem pesquisas válidas."
assert df_fato_chamados.filter(F.col("qtd_pesquisas_validas") > F.col("qtd_pesquisas_recebidas")).count() == 0, "Gold inválida: pesquisas válidas excedem pesquisas recebidas."
assert df_fato_chamados.filter(F.col("fl_espera_valida") & (F.col("tempo_espera_calculado_segundos") < 0)).count() == 0, "Gold inválida: tempo de espera válido negativo."
assert df_fato_chamados.filter(F.col("fl_atendimento_valido") & (F.col("tempo_atendimento_calculado_segundos") < 0)).count() == 0, "Gold inválida: tempo de atendimento válido negativo."

total_custos_silver = df_silver_custos.agg(F.sum("custo").cast("decimal(38,2)").alias("total"))
total_custos_agregado = df_custos_por_chamado.agg(F.sum("custo_total").cast("decimal(38,2)").alias("total"))
total_custos_fato = df_fato_chamados.agg(F.sum("custo_total").cast("decimal(38,2)").alias("total"))
assert total_custos_silver.first()["total"] == total_custos_agregado.first()["total"] == total_custos_fato.first()["total"], "Gold inválida: custos não reconciliam entre Silver, agregado e fato."

qtd_pesquisas_silver = df_silver_pesquisas.select("id_pesquisa").distinct().count()
qtd_pesquisas_fato = df_fato_chamados.agg(F.sum("qtd_pesquisas_recebidas").alias("qtd")).first()["qtd"] or 0
qtd_validas_silver = df_silver_pesquisas.filter(F.col("nota_valida")).count()
qtd_validas_fato = df_fato_chamados.agg(F.sum("qtd_pesquisas_validas").alias("qtd")).first()["qtd"] or 0
assert qtd_pesquisas_silver == qtd_pesquisas_fato, "Gold inválida: pesquisas recebidas não reconciliam."
assert qtd_validas_silver == qtd_validas_fato, "Gold inválida: pesquisas válidas não reconciliam."

df_cobertura_fato = df_fato_chamados.agg(
    F.count("*").alias("qtd_chamados"),
    F.sum(F.col("fl_atendente_cadastrado").cast("int")).alias("qtd_com_atendente"),
    F.sum(F.col("fl_custo_disponivel").cast("int")).alias("qtd_com_custo"),
    F.sum(F.col("fl_pesquisa_valida").cast("int")).alias("qtd_com_pesquisa_valida"),
    F.sum(F.col("data_abertura").isNotNull().cast("int")).alias("qtd_com_data_valida"),
)
display(df_cobertura_fato)

## Agregado executivo mensal

**Grão:** uma linha por `ano_mes`.

As taxas guardam também numerador, denominador e cobertura. Nota média usa `soma_notas_validas / qtd_pesquisas_validas`; satisfação usa respostas 4 ou 5 sobre pesquisas válidas; médias de tempo usam somente chamados com a respectiva flag válida. Toda divisão protege denominador zero.

In [0]:
df_agg_mensal_base = (
    df_fato_chamados
    .filter(F.col("ano_mes_abertura").isNotNull())
    .groupBy(F.col("ano_mes_abertura").alias("ano_mes"))
    .agg(
        F.count("id_chamado").alias("qtd_chamados"),
        F.sum(F.col("fl_resolvido").cast("long")).alias("qtd_resolvidos"),
        F.sum(F.col("fl_resolucao_informada").cast("long")).alias("qtd_resolucao_informada"),
        F.sum("custo_total").cast("decimal(38,2)").alias("custo_total"),
        F.sum(F.col("fl_custo_disponivel").cast("long")).alias("qtd_chamados_com_custo"),
        F.sum(F.coalesce(F.col("qtd_pesquisas_validas"), F.lit(0))).alias("qtd_pesquisas_validas"),
        F.sum(F.coalesce(F.col("soma_notas_validas"), F.lit(0))).alias("soma_notas_validas"),
        F.sum(F.coalesce(F.col("qtd_respostas_satisfeitas"), F.lit(0))).alias("qtd_respostas_satisfeitas"),
        F.sum(F.when(F.col("fl_espera_valida"), F.col("tempo_espera_calculado_segundos")).otherwise(0)).alias("soma_tempo_espera_segundos"),
        F.sum(F.col("fl_espera_valida").cast("long")).alias("qtd_espera_valida"),
        F.sum(F.when(F.col("fl_atendimento_valido"), F.col("tempo_atendimento_calculado_segundos")).otherwise(0)).alias("soma_tempo_atendimento_segundos"),
        F.sum(F.col("fl_atendimento_valido").cast("long")).alias("qtd_atendimento_valido"),
    )
)

def dividir_seguro(numerador: str, denominador: str):
    """Retorna razão nula quando não existe população elegível."""
    return F.when(F.col(denominador) > 0, F.col(numerador) / F.col(denominador))

df_agg_visao_executiva_mensal = (
    df_agg_mensal_base
    .withColumn("taxa_resolucao", dividir_seguro("qtd_resolvidos", "qtd_resolucao_informada"))
    .withColumn("custo_medio_por_chamado_com_custo", dividir_seguro("custo_total", "qtd_chamados_com_custo"))
    .withColumn("taxa_satisfacao", dividir_seguro("qtd_respostas_satisfeitas", "qtd_pesquisas_validas"))
    .withColumn("nota_media", dividir_seguro("soma_notas_validas", "qtd_pesquisas_validas"))
    .withColumn("tempo_medio_espera_segundos", dividir_seguro("soma_tempo_espera_segundos", "qtd_espera_valida"))
    .withColumn("tempo_medio_atendimento_segundos", dividir_seguro("soma_tempo_atendimento_segundos", "qtd_atendimento_valido"))
    .withColumn("cobertura_resolucao", dividir_seguro("qtd_resolucao_informada", "qtd_chamados"))
    .withColumn("cobertura_custo", dividir_seguro("qtd_chamados_com_custo", "qtd_chamados"))
    .withColumn("cobertura_satisfacao", dividir_seguro("qtd_pesquisas_validas", "qtd_chamados"))
    .withColumn("cobertura_tempo_espera", dividir_seguro("qtd_espera_valida", "qtd_chamados"))
    .withColumn("cobertura_tempo_atendimento", dividir_seguro("qtd_atendimento_valido", "qtd_chamados"))
    .orderBy("ano_mes")
)

validar_chave_unica(df_agg_visao_executiva_mensal, tabelas_gold["mensal"], "ano_mes")
assert df_agg_visao_executiva_mensal.agg(F.sum("qtd_chamados").alias("qtd")).first()["qtd"] == df_fato_chamados.filter(F.col("ano_mes_abertura").isNotNull()).count(), "Agregado mensal inválido: volume não reconcilia com a fato."

## Persistência

O projeto recebe snapshots completos. Por isso, `overwrite` é a estratégia explícita para reexecução determinística sem acúmulo de registros. As validações críticas anteriores ocorrem antes das escritas.

In [0]:
dataframes_saida = {
    tabelas_gold["canais"]: df_dim_canais,
    tabelas_gold["motivos"]: df_dim_motivos,
    tabelas_gold["atendentes"]: df_dim_atendentes,
    tabelas_gold["clientes"]: df_dim_clientes,
    tabelas_gold["data"]: df_dim_data,
    tabelas_gold["chamados"]: df_fato_chamados,
    tabelas_gold["mensal"]: df_agg_visao_executiva_mensal,
}

for nome_tabela, df_saida in dataframes_saida.items():
    (
        df_saida.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

## Resumo da execução

Esta seção consolida as tabelas produzidas, seus respectivos grãos, as quantidades de registros e o resultado das validações executadas.

In [0]:
resumo_execucao = [
    (tabelas_gold["canais"], "id_canal", df_dim_canais.count(), "validado", "Canal e etapa do funil"),
    (tabelas_gold["motivos"], "id_motivo", df_dim_motivos.count(), "validado", "Cinco categorias validadas na Silver"),
    (tabelas_gold["atendentes"], "id_atendente", df_dim_atendentes.count(), "validado", "Nome mantido por decisão do grupo"),
    (tabelas_gold["clientes"], "id_cliente", df_dim_clientes.count(), "validado", "Dados cadastrais fictícios mantidos por decisão do grupo"),
    (tabelas_gold["data"], "data", df_dim_data.count(), "validado", "Intervalo de aberturas válidas"),
    (tabelas_gold["chamados"], "id_chamado", df_fato_chamados.count(), "validado", "Custos e pesquisas reconciliados"),
    (tabelas_gold["mensal"], "ano_mes", df_agg_visao_executiva_mensal.count(), "validado", "Indicadores com cobertura"),
]

df_resumo_execucao = spark.createDataFrame(
    resumo_execucao,
    ["tabela", "grao", "quantidade_registros", "status_validacoes", "observacoes"],
)
display(df_resumo_execucao)